In [0]:
# ============================================
# Stage 4.6: Load FactSales (Order Line Grain)
# ============================================

from pyspark.sql import functions as F

raw_base = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/"
cur_base = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/"

# 源数据
df_header  = spark.read.parquet(raw_base + "SalesOrderHeader/")
df_detail  = spark.read.parquet(raw_base + "SalesOrderDetail/")

# 维度表（用于代理键查找）
dim_customer = spark.read.format("delta").load(cur_base + "DimCustomer/")
dim_product  = spark.read.format("delta").load(cur_base + "DimProduct/")
dim_salesp   = spark.read.format("delta").load(cur_base + "DimSalesPerson/")
dim_terr     = spark.read.format("delta").load(cur_base + "DimTerritory/")

print(f"SalesOrderHeader: {df_header.count()}")
print(f"SalesOrderDetail: {df_detail.count()}")

SalesOrderHeader: 31465
SalesOrderDetail: 121317


In [0]:
# JOIN：订单明细 + 订单头 + 各维度（查找代理键）
fact_sales = (df_detail.alias("d")
    .join(df_header.alias("h"), F.col("d.SalesOrderID") == F.col("h.SalesOrderID"), how="inner")
    # 代理键查找——注意加 is_current == 1
    .join(dim_customer.alias("dc"), 
          (F.col("h.CustomerID") == F.col("dc.CustomerID")) & (F.col("dc.is_current") == 1), 
          how="left")
    .join(dim_product.alias("dp"), 
          (F.col("d.ProductID") == F.col("dp.ProductID")) & (F.col("dp.is_current") == 1), 
          how="left")
    .join(dim_salesp.alias("dsp"), 
          (F.col("h.SalesPersonID") == F.col("dsp.SalesPersonID")) & (F.col("dsp.is_current") == 1), 
          how="left")
    .join(dim_terr.alias("dt"), 
          F.col("h.TerritoryID") == F.col("dt.TerritoryID"), 
          how="left")
    .select(
        # 代理键
        F.date_format(F.col("h.OrderDate"), "yyyyMMdd").cast("int").alias("DateKey"),
        F.col("dc.CustomerKey").alias("CustomerKey"),
        F.col("dp.ProductKey").alias("ProductKey"),
        F.col("dsp.SalesPersonKey").alias("SalesPersonKey"),
        F.col("dt.TerritoryKey").alias("TerritoryKey"),
        # 退化维度
        F.col("d.SalesOrderID").alias("SalesOrderID"),
        F.col("d.SalesOrderDetailID").alias("SalesOrderDetailID"),
        # 度量
        F.col("d.OrderQty").cast("int").alias("OrderQty"),
        F.col("d.UnitPrice").cast("decimal(19,4)").alias("UnitPrice"),
        F.col("d.UnitPriceDiscount").cast("decimal(19,4)").alias("UnitPriceDiscount"),
        F.col("d.LineTotal").cast("decimal(19,4)").alias("LineTotal")
    )
)

print(f"FactSales rows: {fact_sales.count()}")
# 期望：121,317

FactSales rows: 121317


In [0]:
# 验证：检查代理键 NULL
fact_sales.select(
    F.sum(F.when(F.col("DateKey").isNull(), 1).otherwise(0)).alias("NullDateKey"),
    F.sum(F.when(F.col("CustomerKey").isNull(), 1).otherwise(0)).alias("NullCustomerKey"),
    F.sum(F.when(F.col("ProductKey").isNull(), 1).otherwise(0)).alias("NullProductKey"),
    F.sum(F.when(F.col("SalesPersonKey").isNull(), 1).otherwise(0)).alias("NullSalesPersonKey"),
    F.sum(F.when(F.col("TerritoryKey").isNull(), 1).otherwise(0)).alias("NullTerritoryKey")
).show()

+-----------+---------------+--------------+------------------+----------------+
|NullDateKey|NullCustomerKey|NullProductKey|NullSalesPersonKey|NullTerritoryKey|
+-----------+---------------+--------------+------------------+----------------+
|          0|              0|             0|             60398|               0|
+-----------+---------------+--------------+------------------+----------------+



In [0]:
# 写入 Curated Zone
curated_path = cur_base + "FactSales/"

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ FactSales written to {curated_path}")

✅ FactSales written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/FactSales/


In [0]:
# 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
# 期望：121317

# 对账：LineTotal 总和
df_check.select(F.sum("LineTotal")).show()
# 应该和 OLTP 的 SUM(LineTotal) 一致

Total rows: 121317
+--------------+
|sum(LineTotal)|
+--------------+
|109846381.4250|
+--------------+

